# 🚀 TopAI — Local AI Playground

### Run open AI models directly inside your notebook runtime.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghostdev102/TopAI/blob/main/ai.ipynb)

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ghostdev102/TopAI/HEAD?labpath=ai.ipynb)

[![GitHub](https://img.shields.io/badge/GitHub-TopAI-black?logo=github)](https://github.com/ghostdev102/TopAI)

---

## 🧠 What is TopAI?

**TopAI** is a free AI playground for experimenting with open models in hosted notebook environments.

This notebook performs **local inference** inside the current runtime.

That means:

- 🔐 No AI API key required
- 💳 No paid inference API required
- 👑 No root access required
- 🖥️ Inference happens inside the notebook runtime
- 🤗 Models come from Hugging Face
- 🎮 GPU is detected automatically when available
- 🐢 CPU fallback is supported
- 💬 Interactive chat interface included
- ⚡ Built-in performance benchmark
- 🎛️ Generation controls

> **Your runtime. Your model. Your inference.**

---

## 🏆 Supported environments

| Environment | Supported |
|---|---|
| Google Colab | ✅ |
| MyBinder | ✅ |
| JupyterLab | ✅ |
| Jupyter Notebook | ✅ |
| Kaggle | ✅ |
| CPU inference | ✅ |
| CUDA GPU | ✅ |

---

## ⚠️ Hardware matters

The model is actually running inside your runtime, so its RAM/VRAM limits apply.

The default model is intentionally small enough to be practical on CPU-only hosted environments. Larger models can be selected if your runtime has enough memory.


# 🔥 1. Runtime Inspector

Let's see what hardware Binder, Colab, or your local Jupyter server gave us.

In [ ]:
import os
import platform
import psutil
import torch

RAM_GB = psutil.virtual_memory().total / (1024 ** 3)
GPU = torch.cuda.is_available()

print("╔══════════════════════════════════════════════════╗")
print("║              🔥 TOPAI RUNTIME                  ║")
print("╚══════════════════════════════════════════════════╝")
print()
print(f"🖥️  Platform       : {platform.platform()}")
print(f"🐍 Python         : {platform.python_version()}")
print(f"🧠 CPU threads    : {os.cpu_count()}")
print(f"💾 System RAM     : {RAM_GB:.2f} GB")
print(f"🔥 PyTorch        : {torch.__version__}")
print(f"🎮 CUDA available : {GPU}")

if GPU:
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"🚀 GPU            : {gpu_name}")
    print(f"💾 VRAM           : {vram:.2f} GB")
    print()
    print("🚀 GPU acceleration is available!")
else:
    print()
    print("🐢 No CUDA GPU detected — using CPU inference.")

print()
print("✅ Runtime inspection complete.")

# 🧠 2. Choose a Model

TopAI can load compatible causal language models from Hugging Face.

### Included presets

| Preset | Model | Intended runtime |
|---|---|---|
| 🟢 Lightweight | Qwen 0.5B | CPU / Binder |
| 🟡 Bigger | Qwen 1.5B | More RAM recommended |
| 🟠 Larger | Qwen 3B | GPU / lots of RAM recommended |

You can also enter any compatible Hugging Face model ID manually.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

MODEL_PRESETS = {
    "🟢 Qwen 0.5B — Binder friendly": "Qwen/Qwen2.5-0.5B-Instruct",
    "🟡 Qwen 1.5B — More capable": "Qwen/Qwen2.5-1.5B-Instruct",
    "🟠 Qwen 3B — Larger": "Qwen/Qwen2.5-3B-Instruct"
}

model_dropdown = widgets.Dropdown(
    options=list(MODEL_PRESETS.keys()),
    value=list(MODEL_PRESETS.keys())[0],
    description="Preset:",
    layout=widgets.Layout(width="700px")
)

custom_model = widgets.Text(
    value="",
    placeholder="Optional: organization/model-name",
    description="Custom:",
    layout=widgets.Layout(width="700px")
)

def selected_model():
    if custom_model.value.strip():
        return custom_model.value.strip()
    return MODEL_PRESETS[model_dropdown.value]

display(model_dropdown)
display(custom_model)

print("Selected:", selected_model())

# 🚀 3. Load the Model

Press the button below to download and load the selected model.

The model is cached in the notebook user's Hugging Face cache so subsequent uses can reuse downloaded files when the runtime persists.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import clear_output

tokenizer = None
model = None
MODEL_NAME = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

load_button = widgets.Button(
    description="🚀 Load Model",
    button_style="success",
    icon="download"
)

load_output = widgets.Output()

def load_model(_):
    global tokenizer, model, MODEL_NAME, DEVICE

    with load_output:
        clear_output()

        MODEL_NAME = selected_model()
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

        print("╔══════════════════════════════════════════════════╗")
        print("║             📥 TOPAI MODEL LOADER              ║")
        print("╚══════════════════════════════════════════════════╝")
        print()
        print("🧠 Model:", MODEL_NAME)
        print("🖥️  Device:", DEVICE)
        print()
        print("📥 Downloading/loading tokenizer...")

        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

        print("📥 Downloading/loading model...")

        if DEVICE == "cuda":
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                torch_dtype=torch.float16,
                device_map="auto"
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                torch_dtype=torch.float32
            )
            model = model.to(DEVICE)

        model.eval()

        print()
        print("╔══════════════════════════════════════════════════╗")
        print("║              ✅ MODEL ONLINE                  ║")
        print("╚══════════════════════════════════════════════════╝")
        print()
        print("🚀 TopAI is ready for inference.")

load_button.on_click(load_model)
display(load_button)
display(load_output)

# 💬 4. Inference Engine

This function performs the actual local generation.

In [ ]:
def generate(
    prompt,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9
):
    if model is None or tokenizer is None:
        raise RuntimeError("Load a model first.")

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

print("✅ Inference engine ready.")

# 🤖 5. First AI Response

Once the model has been loaded, try a prompt.

In [ ]:
if model is None:
    print("⚠️ Load a model in the previous cell first.")
else:
    prompt = "Explain artificial intelligence in three concise sentences."

    print("👤 USER")
    print(prompt)
    print()
    print("🤖 TOPAI — LOCAL INFERENCE")
    print("─" * 60)
    print(generate(prompt))

# ⚡ 6. Benchmark

Measure generation speed on **your actual runtime**.

Results will vary depending on the model, CPU, GPU, RAM, and notebook provider.

In [ ]:
import time

if model is None:
    print("⚠️ Load a model first.")
else:
    benchmark_prompt = "Write a short paragraph explaining why local AI can be useful."

    messages = [{
        "role": "user",
        "content": benchmark_prompt
    }]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    generated_tokens = output.shape[1] - inputs["input_ids"].shape[1]
    speed = generated_tokens / elapsed if elapsed else 0

    print("╔══════════════════════════════════════════════════╗")
    print("║              ⚡ TOPAI BENCHMARK                ║")
    print("╚══════════════════════════════════════════════════╝")
    print()
    print(f"🧠 Model          : {MODEL_NAME}")
    print(f"🖥️  Device         : {DEVICE}")
    print(f"⏱️  Generation time: {elapsed:.2f}s")
    print(f"🧮 New tokens     : {generated_tokens}")
    print(f"🚀 Speed          : {speed:.2f} tokens/sec")
    print()
    print("🏁 Benchmark complete.")

# 💬 7. Interactive Chat

A proper notebook interface — no endless `input()` loop and no terminal required.

Type your prompt, adjust the generation settings, and press **Generate**.

In [ ]:
from IPython.display import display, Markdown, clear_output

prompt_box = widgets.Textarea(
    placeholder="Ask the local model anything...",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="130px")
)

max_tokens = widgets.IntSlider(
    value=256,
    min=32,
    max=1024,
    step=32,
    description="Tokens:",
    continuous_update=False
)

temperature = widgets.FloatSlider(
    value=0.7,
    min=0.0,
    max=1.5,
    step=0.1,
    description="Temp:",
    continuous_update=False
)

generate_button = widgets.Button(
    description="🚀 Generate",
    button_style="success",
    icon="bolt",
    layout=widgets.Layout(width="180px")
)

clear_button = widgets.Button(
    description="🧹 Clear",
    button_style="warning",
    icon="trash",
    layout=widgets.Layout(width="180px")
)

chat_output = widgets.Output()

def generate_from_ui(_):
    with chat_output:
        clear_output()

        if model is None:
            print("⚠️ Load a model before chatting.")
            return

        prompt = prompt_box.value.strip()

        if not prompt:
            print("⚠️ Enter a prompt first.")
            return

        print("🤖 Generating locally...\n")

        try:
            answer = generate(
                prompt,
                max_new_tokens=max_tokens.value,
                temperature=temperature.value
            )

            display(Markdown("### 🤖 TopAI\n\n" + answer))

        except Exception as error:
            print("❌ Generation error:")
            print(error)

def clear_chat(_):
    prompt_box.value = ""
    with chat_output:
        clear_output()

generate_button.on_click(generate_from_ui)
clear_button.on_click(clear_chat)

display(prompt_box)
display(widgets.HBox([max_tokens, temperature]))
display(widgets.HBox([generate_button, clear_button]))
display(chat_output)

# 🧪 8. Quick AI Tests

Run a few prompts to see what your selected model can do.

In [ ]:
TEST_PROMPTS = [
    "Write a Python function that checks whether a number is prime.",
    "Explain recursion to a beginner using a simple analogy.",
    "Write a three-line poem about artificial intelligence.",
    "Give me five ideas for a small AI project."
]

if model is None:
    print("⚠️ Load a model first.")
else:
    for i, prompt in enumerate(TEST_PROMPTS, 1):
        print(f"\n{'=' * 70}")
        print(f"🧪 TEST {i}")
        print(f"👤 {prompt}")
        print(f"\n🤖 {generate(prompt, max_new_tokens=160)}")

# 🏆 9. TopAI Scorecard

You made it.

In [ ]:
print("╔════════════════════════════════════════════════════╗")
print("║                 🏆 TOPAI COMPLETE                 ║")
print("╠════════════════════════════════════════════════════╣")
print("║                                                    ║")
print("║  🧠 Local AI inference             ✅             ║")
print("║  🔐 API key required               ❌             ║")
print("║  💳 Paid inference API required    ❌             ║")
print("║  👑 Root / sudo required           ❌             ║")
print("║  🎮 Automatic GPU detection        ✅             ║")
print("║  🐢 CPU fallback                   ✅             ║")
print("║  🤗 Hugging Face models            ✅             ║")
print("║  💬 Interactive notebook chat      ✅             ║")
print("║  🎛️  Generation controls            ✅             ║")
print("║  ⚡ Performance benchmark           ✅             ║")
print("║  🧪 AI test suite                  ✅             ║")
print("║                                                    ║")
print("║        YOUR RUNTIME. YOUR MODEL. YOUR AI.         ║")
print("║                                                    ║")
print("╚════════════════════════════════════════════════════╝")

# 🌟 What's next?

TopAI is designed to be expanded.

Ideas for future versions:

- More open models
- GGUF / llama.cpp support
- Model download manager
- Conversation history
- Streaming generation
- Vision models
- Speech-to-text
- Text-to-speech
- Image generation
- Free cloud API integrations
- Automatic model recommendations based on available RAM/VRAM
- Model comparison benchmarks

---

## 🔗 TopAI

GitHub: https://github.com/ghostdev102/TopAI

Notebook: `ai.ipynb`

### ⭐ Star the project if you find it useful.